# Testing Notebook

In [1]:
import shared_utils

In [2]:
import _replica_utils

In [3]:
import gcsfs as fs
import geopandas as gpd
import numpy as np
import pandas as pd
from calitp_data_analysis import get_fs, utils
from calitp_data_analysis.sql import to_snakecase
# from siuba import *

fs = get_fs()

In [4]:
pd.set_option("display.max_columns", None)

In [5]:
gcs_path = "gs://calitp-analytics-data/data-analyses/big_data/STM/"

In [6]:
blk_grp_url = "CA_Census_blocks_w_Cities_centered.zip"

In [7]:
shape_data_name = "origins/California_hex7_layer.zip"
origins_name = "D7_8_11_12_replica-stm_regional_travel-06_25_26-trips_dataset.zip"

In [8]:
df = to_snakecase( pd.read_csv(f"{gcs_path}{origins_name}"))  

/tmp/ipykernel_3729/3138358181.py:1: DtypeWarning: Columns (58) have mixed types. Specify dtype option on import or set low_memory=False.
  df = to_snakecase( pd.read_csv(f"{gcs_path}{origins_name}"))


In [9]:
df_origins, df_dests = _replica_utils.prep_replica_data_w_shp(df, shape_data_name, 'origin_custom_id', 'destination_custom_id')

In [10]:
df_origins.sample()

,activity_id,origin_bgrp_2020,origin_trct_2020,origin_cty_2020,origin_st_2020,destination_bgrp_2020,destination_trct_2020,destination_cty_2020,destination_st_2020,primary_mode,trip_purpose,previous_trip_purpose,trip_start_time,trip_end_time,trip_duration_minutes,trip_distance_miles,vehicle_type,vehicle_fuel_type,transit_submode,transit_agency,transit_route,origin_land_use,origin_building_use,destination_land_use,destination_building_use,trip_taker_person_id,trip_taker_household_id,trip_taker_age,trip_taker_sex,trip_taker_race_ethnicity,trip_taker_employment_status,trip_taker_wfh,trip_taker_individual_income,trip_taker_commute_mode,trip_taker_household_size,trip_taker_household_income,trip_taker_available_vehicles,trip_taker_resident_type,trip_taker_industry,trip_taker_building_type,trip_taker_school_grade_attending,trip_taker_education,trip_taker_tenure,trip_taker_language,trip_taker_home_bgrp_2020,trip_taker_home_trct_2020,trip_taker_home_cty_2020,trip_taker_home_st_2020,trip_taker_work_bgrp_2020,trip_taker_work_trct_2020,trip_taker_work_cty_2020,trip_taker_work_st_2020,origin_custom,destination_custom,origin_custom_id,origin_bgrp_fips_2020,origin_custom_lng,origin_custom_lat,destination_bgrp_fips_2020,destination_custom_id,destination_custom_lat,destination_custom_lng,origin_geometry
110641,1349651961432132899,"1 (Tract 9800.13, Los Angeles, CA)","9800.13 (Los Angeles, CA)","Los Angeles County, CA",California,"1 (Tract 23.02, Clark, NV)","23.02 (Clark, NV)","Clark County, NV",Nevada,auto_passenger,shop,school,17:01:00,22:11:10,310,276.3,NaN,unknown_fuel_type,NaN,NaN,NaN,education,education,mixed_use,retail,3469475222113737137,372632050043710127,19.0,male,black_not_hispanic_or_latino,employed,employed_not_working,2849.0,auto_passenger,2,65767.0,one,core,naics721110,several_units,school,k_12,renter,english,"2 (Tract 6006.01, Los Angeles, CA)","6006.01 (Los Angeles, CA)","Los Angeles County, CA",California,"1 (Tract 2079.01, Los Angeles, CA)","2079.01 (Los Angeles, CA)","Los Angeles County, CA",California,8729a565affffff,872986b83ffffff,608718595333029900,60379800131,-118.3875,33.9229,320030023021,6.087165e+17,36.1376,-115.1728,"POLYGON ((-118.37921 33.91162, -118.39535 33.9..."


In [11]:
# with get_fs().open(f"{gcs_path}{place_data}") as f:
#         plc = to_snakecase(gpd.read_file(f))

In [12]:
# plc.sample()

In [13]:
# plc.columns

In [14]:
# plc = plc[['objectid', 'state', 'geoid', 'county', 'tract', 'blkgrp', 'name', 'basename', 'cdtfa_coun', 'city_name', 'geometry']]

In [15]:
# ### add in a period and ", CA" to match the format in the replica data 
# plc['tract'] = plc['tract'].str[:-2] + '.' + plc['tract'].str[-2:]
# plc['county_name'] = plc['cdtfa_coun'] + ", CA"


In [16]:
# plc['city_name'] = plc['city_name'].fillna('Unincorporated')

In [17]:


# ### Format the block group column by removing trailing and leading 0s
# ### Replica's data format is different for tract numbers
# ### Ex: 00111.00 vs 111
# condition = plc['tract'].str.endswith(".00")
# plc['tract'] = np.where(condition, plc['tract'].str.replace(".00", ""), plc['tract'])

# plc['corrected_tract'] = plc['tract'].str.lstrip('00')
# plc['corrected_tract'] = plc['tract'].str.lstrip('0')

In [18]:
# plc['city_county'] = plc['city_name'] + ', ' + plc['cdtfa_coun']

In [19]:
# plc.sample(10)

In [20]:
def prep_place_data(place_data_df, county_name_col, tract_col, city_name_col, state):

    ### add in a period and ", CA" to match the format in the replica data 
    place_data_df[tract_col] = place_data_df[tract_col].str[:-2] + '.' + place_data_df['tract'].str[-2:]
    place_data_df['county_name'] = place_data_df[[county_name_col]] + f", {state}"

    ### replace the na locations
    place_data_df[city_name_col] = place_data_df[city_name_col].fillna('Unincorporated')

    ### Format the block group column by removing trailing and leading 0s
    ### Replica's data format is different for tract numbers
    ### Ex: 00111.00 vs 111
    condition = place_data_df[tract_col].str.endswith(".00")
    place_data_df[tract_col] = np.where(condition, place_data_df[tract_col].str.replace(".00", ""), place_data_df['tract'])
    
    place_data_df['corrected_tract'] = place_data_df[tract_col].str.lstrip('00')
    place_data_df['corrected_tract'] = place_data_df[tract_col].str.lstrip('0')
    
    ### add together the county and city names, helpful for thing like "Unincorporated, County"
    place_data_df['city_county'] = place_data_df[city_name_col] + ', ' + place_data_df[county_name_col]

    return place_data_df

In [21]:
def read_and_prep_place_data(ca_place_path, nv_place_path,):
    
    ## read in the place data for CA
    with get_fs().open(f"{gcs_path}{ca_place_path}") as f:
        ca_plc = to_snakecase(gpd.read_file(f))
    ## susbet data
    ca_plc = ca_plc[['objectid', 'state', 'geoid', 'county', 'tract', 'blkgrp', 'name', 'basename', 'cdtfa_coun', 'city_name', 'geometry']]
    ## format
    ca = prep_place_data(ca_plc, county_name_col='cdtfa_coun', tract_col='tract', city_name_col='city_name', state="CA")

    ### repreat for NV
    with get_fs().open(f"{gcs_path}{nv_place_path}") as f:
        nv_plc = to_snakecase(gpd.read_file(f))
    nv_plc['countyname'] = nv_plc['countyname'] + " County"
    nv = prep_place_data(nv_plc, county_name_col='countyname', tract_col='tract', city_name_col='city_name', state="NV")

    nv_subset = nv[['corrected_tract', 'blkgrp','county_name', 'city_county']] 
    ca_subset = ca[['corrected_tract', 'blkgrp','county_name', 'city_county']]

    places = pd.concat([ca_subset, nv_subset], axis=0, ignore_index=True)

    return places

In [22]:
def add_cities_to_origin_dest(df, ca_place_path, nv_place_path, origin_county_col, orgin_tract_col, origin_blkgrp_col, dest_county_col, dest_tract_col, dest_blkgrp_col):

    places = read_and_prep_place_data(ca_place_path, nv_place_path)

    ### set up replica data by extracting just the tract number and blockgroup number 
    df['origin_tract'] = df[orgin_tract_col].str.split(' (', regex=False).str[0]
    df['origin_blkgrp'] = df[origin_blkgrp_col].str.split(' (', regex=False).str[0]
    
    df['dest_tract'] = df[dest_tract_col].str.split(' (', regex=False).str[0]
    df['dest_blkgrp'] = df[dest_blkgrp_col].str.split(' (', regex=False).str[0]

    ### merge together! 
    ### first merge origins to get the origin city and then merge the destinations to get destination city

    df2 = pd.merge(
        df, 
        places[['county_name', 'corrected_tract', 'blkgrp', 'city_county']],  
        left_on=[origin_county_col, 'origin_tract', 'origin_blkgrp'], 
        right_on=['county_name', 'corrected_tract', 'blkgrp'],
        how='left'
    )

    ### drop the blk_grps columms for the second merge and rename city col to distinguish
    df2 = df2.drop(columns=['county_name', 'corrected_tract', 'blkgrp'])
    df2 = df2.rename(columns={"city_county":"origin_city"})
    
    ### and repeat
    df2 = pd.merge(
        df2, 
       places[['county_name', 'corrected_tract', 'blkgrp', 'city_county']], 
        left_on=[dest_county_col, 'dest_tract', 'dest_blkgrp'], 
        right_on=['county_name', 'corrected_tract', 'blkgrp'],
        how='left'
    )
    
    df2 = df2.drop(columns=['county_name', 'corrected_tract', 'blkgrp'])
    df2 = df2.rename(columns={"city_county":"dest_city"})

    df2['dest_city'] = df2['dest_city'].fillna('Out of Region')

    return df2

In [23]:
ca_place_data = "CA_Census_Block_with_Places.zip"

In [24]:
nv_place_data = "NV_Census_Block_with_Places.zip"

In [25]:
df2 = add_cities_to_origin_dest(df_origins, ca_place_data, nv_place_data, "origin_cty_2020", "origin_trct_2020", "origin_bgrp_2020", "destination_cty_2020", "destination_trct_2020", "destination_bgrp_2020")

In [26]:
# ## read in the data
# with get_fs().open(f"{gcs_path}{ca_place_data}") as f:
#     ca_plc = to_snakecase(gpd.read_file(f))

# ca_plc = ca_plc[['objectid', 'state', 'geoid', 'county', 'tract', 'blkgrp', 'name', 'basename', 'cdtfa_coun', 'city_name', 'geometry']]

In [27]:
# test_ca = prep_place_data(ca_plc, county_name_col='cdtfa_coun', tract_col='tract', city_name_col='city_name', state="CA")

In [28]:
# test_ca.sample()

In [29]:
# with get_fs().open(f"{gcs_path}{nv_place_data}") as f:
#     nv_plc = to_snakecase(gpd.read_file(f))

In [30]:
# nv_plc['countyname'] = nv_plc['countyname'] + " County"

In [31]:
# test_nv = prep_place_data(nv_plc, county_name_col='countyname', tract_col='tract', city_name_col='city_name', state="NV")

In [32]:
# test_nv.sample()

In [33]:
# nv_subset = test_nv[['corrected_tract', 'blkgrp','county_name', 'city_county']] 
# ca_subset = test_ca[['corrected_tract', 'blkgrp','county_name', 'city_county']]

In [34]:
# places = pd.concat([ca_subset, nv_subset], axis=0, ignore_index=True)

In [35]:
# places

In [36]:
# df_origins.sample()

In [37]:
# ### set up replica data by extracting just the tract number and blockgroup number 
# df_origins['origin_tract'] = df_origins['origin_trct_2020'].str.split(' (', regex=False).str[0]
# df_origins['origin_blkgrp'] = df_origins['origin_bgrp_2020'].str.split(' (', regex=False).str[0]
    
# df_origins['dest_tract'] = df_origins['destination_trct_2020'].str.split(' (', regex=False).str[0]
# df_origins['dest_blkgrp'] = df_origins['destination_bgrp_2020'].str.split(' (', regex=False).str[0]



In [38]:
# df_origins.sample()

In [39]:
# ### merge together! 
# ### first merge origins to get the origin city and then merge the destinations to get destination city

# df2 = pd.merge(
#         df_origins, 
#         places[['county_name', 'corrected_tract', 'blkgrp', 'city_county']], 
#         left_on=['origin_cty_2020', 'origin_tract', 'origin_blkgrp'], 
#         right_on=['county_name', 'corrected_tract', 'blkgrp'],
#         how='left'
#     )


In [40]:
# ### drop the blk_grps columms for the second merge and rename city col to distinguish
# df2 = df2.drop(columns=['county_name', 'corrected_tract', 'blkgrp'])
# df2 = df2.rename(columns={"city_county":"origin_city"})
    

In [41]:
# ### and repeat
# df2 = pd.merge(
#         df2, 
#         places[['county_name', 'corrected_tract', 'blkgrp', 'city_county']], 
#         left_on=['destination_cty_2020', 'dest_tract', 'dest_blkgrp'], 
#         right_on=['county_name', 'corrected_tract', 'blkgrp'],
#         how='left'
#     )

In [42]:
# df2 = df2.drop(columns=['county_name', 'corrected_tract', 'blkgrp'])
# df2 = df2.rename(columns={"city_county":"dest_city"})

In [43]:
# df2['dest_city'] = df2['dest_city'].fillna('Out of Region')

In [44]:
df2

,activity_id,origin_bgrp_2020,origin_trct_2020,origin_cty_2020,origin_st_2020,destination_bgrp_2020,destination_trct_2020,destination_cty_2020,destination_st_2020,primary_mode,trip_purpose,previous_trip_purpose,trip_start_time,trip_end_time,trip_duration_minutes,trip_distance_miles,vehicle_type,vehicle_fuel_type,transit_submode,transit_agency,transit_route,origin_land_use,origin_building_use,destination_land_use,destination_building_use,trip_taker_person_id,trip_taker_household_id,trip_taker_age,trip_taker_sex,trip_taker_race_ethnicity,trip_taker_employment_status,trip_taker_wfh,trip_taker_individual_income,trip_taker_commute_mode,trip_taker_household_size,trip_taker_household_income,trip_taker_available_vehicles,trip_taker_resident_type,trip_taker_industry,trip_taker_building_type,trip_taker_school_grade_attending,trip_taker_education,trip_taker_tenure,trip_taker_language,trip_taker_home_bgrp_2020,trip_taker_home_trct_2020,trip_taker_home_cty_2020,trip_taker_home_st_2020,trip_taker_work_bgrp_2020,trip_taker_work_trct_2020,trip_taker_work_cty_2020,trip_taker_work_st_2020,origin_custom,destination_custom,origin_custom_id,origin_bgrp_fips_2020,origin_custom_lng,origin_custom_lat,destination_bgrp_fips_2020,destination_custom_id,destination_custom_lat,destination_custom_lng,origin_geometry,origin_tract,origin_blkgrp,dest_tract,dest_blkgrp,origin_city,dest_city
0,12085915017420060557,"1 (Tract 1021.04, Los Angeles, CA)","1021.04 (Los Angeles, CA)","Los Angeles County, CA",California,"2 (Tract 11, Kings, CA)","11 (Kings, CA)","Kings County, CA",California,private_auto,home,work,21:55:34,00:49:44,174,178.3,NaN,NaN,NaN,NaN,NaN,single_family,single_family,single_family,single_family,18128471139408472620,10616512179847944618,38.0,female,hispanic_or_latino_origin,not_in_labor_force,unemployed_under_16_not_in_labor_force,12108.0,other_travel_mode,7,12108.0,one,core,naics81,single_family,not_attending_school,high_school,renter,spanish,"2 (Tract 11, Kings, CA)","11 (Kings, CA)","Kings County, CA",California,Does not have work/school location,Does not have work/school location,Does not have work/school location,Does not have work/school location,8729a1892ffffff,8729a80d1ffffff,608718329984581600,60371021041,-118.3445,34.2028,060310011002,6.087188e+17,36.3205,-119.6457,"POLYGON ((-118.33614 34.19148, -118.35232 34.1...",1021.04,1,11,2,"Los Angeles, Los Angeles County","Hanford, Kings County"
1,12616432462675974912,"1 (Tract 1021.04, Los Angeles, CA)","1021.04 (Los Angeles, CA)","Los Angeles County, CA",California,"1 (Tract 1114.01, Los Angeles, CA)","1114.01 (Los Angeles, CA)","Los Angeles County, CA",California,public_transit,shop,recreation,15:08:00,17:49:47,161,37.0,unknown_vehicle_type,unknown_fuel_type,bus,Metro - Los Angeles,Metro Local Line,office,office,office,office,14814586594263718632,9115706970684435042,40.0,male,hispanic_or_latino_origin,employed,in_person,4078.0,private_auto,1,4078.0,zero,core,naics56,multiple_units,not_attending_school,k_12,renter,spanish,"1 (Tract 1275.20, Los Angeles, CA)","1275.20 (Los Angeles, CA)","Los Angeles County, CA",California,"3 (Tract 3104, Los Angeles, CA)","3104 (Los Angeles, CA)","Los Angeles County, CA",California,8729a1892ffffff,8729a1125ffffff,608718329984581600,60371021041,-118.3445,34.2028,060371114011,6.087183e+17,34.2670,-118.4915,"POLYGON ((-118.33614 34.19148, -118.35232 34.1...",1021.04,1,1114.01,1,"Los Angeles, Los Angeles County","Los Angeles, Los Angeles County"
2,12754155054627986973,"1 (Tract 1021.04, Los Angeles, CA)","1021.04 (Los Angeles, CA)","Los Angeles County, CA",California,"3 (Tract 1212.22, Los Angeles, CA)","1212.22 (Los Angeles, CA)","Los Angeles County, CA",California,public_transit,home,work,22:32:00,00:04:19,92,11.4,unknown_vehicle_type,unknown_fuel_type,"bus, bus, bus","Metro - Los Angeles, Metro - Los Angeles, Metr...","Metro Local Line, Metro Local Line, Metro Loca...",education,education,single_family,single_family,11914740138224311444,15226853050545306300,26.0,

In [47]:
# df2[df2["destination_cty_2020"].str.contains("NV")]